# nb_measures_financial

Adds KPI, ratio, and analysis measures to the Direct Lake model **NVIDIA Financials** (host table **Balance Sheet**, folders `02-kpi`, `03-ratios`, `04-analysis`). Idempotent: add or update only measures this notebook owns. No secrets.

**Kernel:** **Python** (not PySpark).

**Before Run all**

1. Attach default lakehouse **NVIDIALakehouse** (needed to read gold Delta for the independent check).
2. Model **NVIDIA Financials** already exists (run `nb_semantic_model.ipynb` first) and gold date columns are Date.
3. Set the kernel to **Python**. XMLA **read/write** must be enabled for the TOM cell.
4. Definitions: `docs/measures.md`.


In [ ]:
%pip install -q semantic-link-labs "deltalake>=1.1"


In [ ]:
# Parameters
WORKSPACE = "NVIDIA-BalanceSheet-AI"
LAKEHOUSE = "NVIDIALakehouse"
DATASET = "NVIDIA Financials"
FY_CHECK = 2026
HOST = "Balance Sheet"


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model
from sempy_labs import refresh_semantic_model
from sempy_labs.lakehouse import lakehouse_attached

MEASURES = json.loads(r'''[
  {
    "name": "Gross Profit",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration GrossProfit at the latest period_end (is_latest_filing). Not a sum of quarters.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"GrossProfit\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)"
  },
  {
    "name": "Operating Income",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration OperatingIncomeLoss at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"OperatingIncomeLoss\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)"
  },
  {
    "name": "Cost of Revenue",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Prefers CostOfRevenue, else CostOfGoodsAndServicesSold, at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _primary =\n    CALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"CostOfRevenue\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)\nVAR _fallback =\n    CALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"CostOfGoodsAndServicesSold\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)\nRETURN\nCOALESCE(_primary, _fallback)"
  },
  {
    "name": "Current Assets",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time AssetsCurrent at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"AssetsCurrent\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Current Liabilities",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time LiabilitiesCurrent at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"LiabilitiesCurrent\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Working Capital",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Current Assets minus Current Liabilities (both point-in-time, latest period).",
    "expr": "[Current Assets] - [Current Liabilities]"
  },
  {
    "name": "Inventory",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time InventoryNet at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"InventoryNet\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Accounts Receivable",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time AccountsReceivableNetCurrent at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"AccountsReceivableNetCurrent\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Marketable Securities",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time MarketableSecuritiesCurrent at the latest period_end. NVIDIA stopped tagging this concept from the FY2026 10-K (last value Q3 FY2026, 2025-10-26), so it is BLANK from FY2026 on — companyfacts has no replacement us-gaap concept.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"MarketableSecuritiesCurrent\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "R&D Expense",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration ResearchAndDevelopmentExpense at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"ResearchAndDevelopmentExpense\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)"
  },
  {
    "name": "EPS Diluted",
    "folder": "02-kpi",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration EarningsPerShareDiluted (USD/shares) at the latest period_end (is_latest_filing).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Income Statement'[value_usd]),\n    'Income Statement'[concept] = \"EarningsPerShareDiluted\",\n    'Income Statement'[is_latest_filing] = TRUE(),\n    'Income Statement'[period_end] = _end\n)"
  },
  {
    "name": "Share Repurchases",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration PaymentsForRepurchaseOfCommonStock at the latest period_end (is_latest_filing). 10-Q may be YTD; see [Cash Flow Is YTD].",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Cash Flow'[period_end]),\n        'Cash Flow'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Cash Flow'[value_usd]),\n    'Cash Flow'[concept] = \"PaymentsForRepurchaseOfCommonStock\",\n    'Cash Flow'[is_latest_filing] = TRUE(),\n    'Cash Flow'[period_end] = _end\n)"
  },
  {
    "name": "Dividends Paid",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration PaymentsOfDividends at the latest period_end (is_latest_filing). Uses PaymentsOfDividends (PaymentsOfDividendsCommonStock is absent from NVIDIA companyfacts). 10-Q may be YTD.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Cash Flow'[period_end]),\n        'Cash Flow'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Cash Flow'[value_usd]),\n    'Cash Flow'[concept] = \"PaymentsOfDividends\",\n    'Cash Flow'[is_latest_filing] = TRUE(),\n    'Cash Flow'[period_end] = _end\n)"
  },
  {
    "name": "Cash from Investing",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration NetCashProvidedByUsedInInvestingActivities at the latest period_end (is_latest_filing). 10-Q may be YTD.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Cash Flow'[period_end]),\n        'Cash Flow'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Cash Flow'[value_usd]),\n    'Cash Flow'[concept] = \"NetCashProvidedByUsedInInvestingActivities\",\n    'Cash Flow'[is_latest_filing] = TRUE(),\n    'Cash Flow'[period_end] = _end\n)"
  },
  {
    "name": "Cash from Financing",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Duration NetCashProvidedByUsedInFinancingActivities at the latest period_end (is_latest_filing). 10-Q may be YTD.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Cash Flow'[period_end]),\n        'Cash Flow'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Cash Flow'[value_usd]),\n    'Cash Flow'[concept] = \"NetCashProvidedByUsedInFinancingActivities\",\n    'Cash Flow'[is_latest_filing] = TRUE(),\n    'Cash Flow'[period_end] = _end\n)"
  },
  {
    "name": "Long-Term Debt Current",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Point-in-time LongTermDebtCurrent (current portion of long-term debt) at the latest period_end (is_latest_filing). Tagged by NVIDIA from FY2020; earlier periods are BLANK.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] = \"LongTermDebtCurrent\",\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Gross Debt",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Long-Term Debt (noncurrent) + Long-Term Debt Current. Excludes operating lease liabilities (see [Operating Lease Liabilities]). BLANK current portion counts as 0; BLANK when noncurrent debt is BLANK.",
    "expr": "VAR _lt = [Long-Term Debt]\nVAR _cur = [Long-Term Debt Current]\nRETURN\nIF(ISBLANK(_lt), BLANK(), _lt + COALESCE(_cur, 0))"
  },
  {
    "name": "Operating Lease Liabilities",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "OperatingLeaseLiabilityCurrent + OperatingLeaseLiabilityNoncurrent at the latest period_end (is_latest_filing). Shown separately from debt; never netted into Net Cash.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nRETURN\nCALCULATE(\n    SUM('Balance Sheet'[value_usd]),\n    'Balance Sheet'[concept] IN { \"OperatingLeaseLiabilityCurrent\", \"OperatingLeaseLiabilityNoncurrent\" },\n    'Balance Sheet'[is_latest_filing] = TRUE(),\n    'Balance Sheet'[period_end] = _end\n)"
  },
  {
    "name": "Net Cash",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Cash & equivalents + current Marketable Securities − Gross Debt (leases excluded). BLANK when Marketable Securities is not tagged (FY2026 onward) so the figure is never understated; use [Net Cash (Cash Only)] as the conservative fallback.",
    "expr": "VAR _ms = [Marketable Securities]\nVAR _debt = [Gross Debt]\nRETURN\nIF(ISBLANK(_ms) || ISBLANK([Cash]), BLANK(), [Cash] + _ms - COALESCE(_debt, 0))"
  },
  {
    "name": "Net Cash (Cash Only)",
    "folder": "02-kpi",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Conservative net cash: Cash & equivalents − Gross Debt (marketable securities and leases excluded). Always available; label it as cash-only on the canvas.",
    "expr": "VAR _cash = [Cash]\nRETURN\nIF(ISBLANK(_cash), BLANK(), _cash - COALESCE([Gross Debt], 0))"
  },
  {
    "name": "Period Label",
    "folder": "02-kpi",
    "format": "",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Text label for the latest balance-sheet period in context, e.g. \"Q2 FY2027 · ended 2026-07-26\" or \"FY2026 · ended 2026-01-25\". Fiscal labels only — never calendar.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), REMOVEFILTERS('Date'), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), REMOVEFILTERS('Date'), 'Date'[date] = _end)\nVAR _is_fy_end =\n    CALCULATE(MAXX(VALUES('Date'[is_fiscal_year_end]), IF('Date'[is_fiscal_year_end], 1, 0)), REMOVEFILTERS('Date'), 'Date'[date] = _end) = 1\nRETURN\nIF(\n    ISBLANK(_end),\n    BLANK(),\n    IF(_is_fy_end, \"FY\" & _fy, \"Q\" & _fq & \" FY\" & _fy) & \" · ended \" & FORMAT(_end, \"yyyy-mm-dd\")\n)"
  },
  {
    "name": "Average Assets",
    "folder": "03-ratios",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Average of ending and prior-period-end (prior FY end at year grain, prior fiscal quarter end at quarter grain) Total Assets (prior quarter wraps Q1 → prior FY Q4). If the prior balance is missing, uses ending Total Assets.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Assets]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _annual =\n    DATEDIFF(MIN('Date'[date]), MAX('Date'[date]), DAY) > 120\nVAR _prior_fy = IF(_annual || _fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_annual || _fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Assets],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_prior), _curr, DIVIDE(_curr + _prior, 2))"
  },
  {
    "name": "Average Equity",
    "folder": "03-ratios",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Average of ending and prior-period-end (prior FY end at year grain, prior fiscal quarter end at quarter grain) Total Equity (Q1 wraps to prior FY Q4). If the prior balance is missing, uses ending Total Equity.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Equity]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _annual =\n    DATEDIFF(MIN('Date'[date]), MAX('Date'[date]), DAY) > 120\nVAR _prior_fy = IF(_annual || _fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_annual || _fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Equity],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_prior), _curr, DIVIDE(_curr + _prior, 2))"
  },
  {
    "name": "Average Inventory",
    "folder": "03-ratios",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Average of ending and prior-period-end (prior FY end at year grain, prior fiscal quarter end at quarter grain) Inventory. Falls back to ending Inventory if the prior period is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Inventory]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _annual =\n    DATEDIFF(MIN('Date'[date]), MAX('Date'[date]), DAY) > 120\nVAR _prior_fy = IF(_annual || _fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_annual || _fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Inventory],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_prior), _curr, DIVIDE(_curr + _prior, 2))"
  },
  {
    "name": "Average Receivables",
    "folder": "03-ratios",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Average of ending and prior-period-end (prior FY end at year grain, prior fiscal quarter end at quarter grain) Accounts Receivable. Falls back to ending AR if the prior period is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Accounts Receivable]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _annual =\n    DATEDIFF(MIN('Date'[date]), MAX('Date'[date]), DAY) > 120\nVAR _prior_fy = IF(_annual || _fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_annual || _fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Accounts Receivable],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_prior), _curr, DIVIDE(_curr + _prior, 2))"
  },
  {
    "name": "Period Days",
    "folder": "03-ratios",
    "format": "#,##0.0",
    "table": "Balance Sheet",
    "hidden": true,
    "description": "Duration_days of the latest income-statement period (is_latest_filing); 365 if blank. Used as the day count for DIO/DSO (NVIDIA FY is typically 363, not 365).",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nRETURN\nCOALESCE(_days, 365)"
  },
  {
    "name": "Gross Margin %",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Gross Profit / Revenue for the latest period in context.",
    "expr": "DIVIDE([Gross Profit], [Revenue])"
  },
  {
    "name": "Operating Margin %",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Operating Income / Revenue for the latest period in context.",
    "expr": "DIVIDE([Operating Income], [Revenue])"
  },
  {
    "name": "Net Margin %",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Net Income / Revenue for the latest period in context. Same definition as 01-base [Net Income Margin %].",
    "expr": "DIVIDE([Net Income], [Revenue])"
  },
  {
    "name": "R&D % of Revenue",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "R&D Expense / Revenue for the latest period in context.",
    "expr": "DIVIDE([R&D Expense], [Revenue])"
  },
  {
    "name": "ROE",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Net Income / Average Equity. Average is (ending + prior-period-end) / 2, prior FY end at year grain, or ending if prior is missing. Not annualized in a quarterly context.",
    "expr": "DIVIDE([Net Income], [Average Equity])"
  },
  {
    "name": "ROA",
    "folder": "03-ratios",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Net Income / Average Assets. Average is (ending + prior-period-end) / 2, prior FY end at year grain, or ending if prior is missing. Not annualized in a quarterly context.",
    "expr": "DIVIDE([Net Income], [Average Assets])"
  },
  {
    "name": "Current Ratio",
    "folder": "03-ratios",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Current Assets / Current Liabilities (point-in-time, latest period).",
    "expr": "DIVIDE([Current Assets], [Current Liabilities])"
  },
  {
    "name": "Quick Ratio",
    "folder": "03-ratios",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "(Cash + Marketable Securities + Accounts Receivable) / Current Liabilities. BLANK when Marketable Securities is not tagged (FY2026 onward) — treating it as 0 would understate NVIDIA's liquidity by tens of billions.",
    "expr": "VAR _ms = [Marketable Securities]\nRETURN\nIF(\n    ISBLANK(_ms),\n    BLANK(),\n    DIVIDE([Cash] + _ms + [Accounts Receivable], [Current Liabilities])\n)"
  },
  {
    "name": "Cash Ratio",
    "folder": "03-ratios",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Cash / Current Liabilities (point-in-time, latest period).",
    "expr": "DIVIDE([Cash], [Current Liabilities])"
  },
  {
    "name": "Liabilities to Assets",
    "folder": "03-ratios",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Total Liabilities / Total Assets (point-in-time, latest period).",
    "expr": "DIVIDE([Total Liabilities], [Total Assets])"
  },
  {
    "name": "Equity Multiplier",
    "folder": "03-ratios",
    "format": "0.00x",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Average Assets / Average Equity (same average convention as ROE/ROA, so DuPont tiles).",
    "expr": "DIVIDE([Average Assets], [Average Equity])"
  },
  {
    "name": "Asset Turnover",
    "folder": "03-ratios",
    "format": "0.00x",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Revenue / Average Assets (same average convention as ROA).",
    "expr": "DIVIDE([Revenue], [Average Assets])"
  },
  {
    "name": "Inventory Turnover",
    "folder": "03-ratios",
    "format": "0.00x",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Cost of Revenue / Average Inventory.",
    "expr": "DIVIDE([Cost of Revenue], [Average Inventory])"
  },
  {
    "name": "DIO",
    "folder": "03-ratios",
    "format": "#,##0.0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Days inventory outstanding: Period Days × Average Inventory / Cost of Revenue. Period Days is the filing duration (not a fixed 365).",
    "expr": "DIVIDE([Period Days] * [Average Inventory], [Cost of Revenue])"
  },
  {
    "name": "DSO",
    "folder": "03-ratios",
    "format": "#,##0.0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Days sales outstanding: Period Days × Average Receivables / Revenue. Period Days is the filing duration (not a fixed 365).",
    "expr": "DIVIDE([Period Days] * [Average Receivables], [Revenue])"
  },
  {
    "name": "CFO to Net Income",
    "folder": "03-ratios",
    "format": "0.00",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Operating Cash Flow / Net Income (cash conversion) on matching windows only. BLANK when cash flow is 10-Q YTD but Net Income is a standalone quarter (Q2/Q3), so YTD cash is never divided by one quarter of earnings.",
    "expr": "VAR _cf_ytd = [Cash Flow Is YTD]\nVAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _inc_ytd =\n    CALCULATE(\n        MAXX(VALUES('Income Statement'[is_ytd]), IF('Income Statement'[is_ytd], 1, 0)),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nRETURN\nIF(\n    _cf_ytd = 1 && COALESCE(_inc_ytd, 0) = 0,\n    BLANK(),\n    DIVIDE([Operating Cash Flow], [Net Income])\n)"
  },
  {
    "name": "Revenue Standalone Quarter",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": true,
    "description": "Hidden helper: Revenue for one standalone fiscal quarter. On Q1–Q3 it is [Revenue]; on the FY 10-K it derives Q4 = FY − Q1 − Q2 − Q3 (BLANK unless all three quarters exist). Used by QoQ and TTM so a quarter is never compared to a full year.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _fy_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nVAR _q_ends =\n    CALCULATETABLE(\n        VALUES('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[fiscal_quarter] IN { 1, 2, 3 },\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _q123 =\n    ADDCOLUMNS(\n        _q_ends,\n        \"@v\",\n            VAR _d = 'Date'[date]\n            RETURN CALCULATE([Revenue], REMOVEFILTERS('Date'), 'Date'[date] = _d)\n    )\nVAR _n = COUNTROWS(FILTER(_q123, NOT ISBLANK([@v])))\nVAR _annual_value =\n    CALCULATE([Revenue], REMOVEFILTERS('Date'), 'Date'[date] = _fy_end)\nRETURN\nIF(\n    _annual,\n    IF(_n = 3 && NOT ISBLANK(_annual_value), _annual_value - SUMX(_q123, [@v])),\n    [Revenue]\n)"
  },
  {
    "name": "Net Income Standalone Quarter",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": true,
    "description": "Hidden helper: Net Income for one standalone fiscal quarter. On Q1–Q3 it is [Net Income]; on the FY 10-K it derives Q4 = FY − Q1 − Q2 − Q3 (BLANK unless all three quarters exist). Used by QoQ and TTM so a quarter is never compared to a full year.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _fy_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nVAR _q_ends =\n    CALCULATETABLE(\n        VALUES('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[fiscal_quarter] IN { 1, 2, 3 },\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _q123 =\n    ADDCOLUMNS(\n        _q_ends,\n        \"@v\",\n            VAR _d = 'Date'[date]\n            RETURN CALCULATE([Net Income], REMOVEFILTERS('Date'), 'Date'[date] = _d)\n    )\nVAR _n = COUNTROWS(FILTER(_q123, NOT ISBLANK([@v])))\nVAR _annual_value =\n    CALCULATE([Net Income], REMOVEFILTERS('Date'), 'Date'[date] = _fy_end)\nRETURN\nIF(\n    _annual,\n    IF(_n = 3 && NOT ISBLANK(_annual_value), _annual_value - SUMX(_q123, [@v])),\n    [Net Income]\n)"
  },
  {
    "name": "Revenue YoY Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Revenue minus the same NVIDIA fiscal quarter in the prior fiscal year (Date[fiscal_year]/[fiscal_quarter] quarter-end). Not SAMEPERIODLASTYEAR. BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Revenue]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Revenue QoQ Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Revenue minus the prior fiscal quarter (Q1 wraps to prior FY Q4, derived as FY − Q1 − Q2 − Q3 via [Revenue Standalone Quarter] so a quarter is never compared to a full year). BLANK when the latest period is the annual 10-K, or when the prior quarter is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Revenue]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Revenue Standalone Quarter],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(_annual || ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Revenue QoQ %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "QoQ change in Revenue as a % of the prior fiscal quarter (Q1 wraps to prior FY Q4, derived as FY − Q1 − Q2 − Q3 via [Revenue Standalone Quarter] so a quarter is never compared to a full year). BLANK when the latest period is the annual 10-K, or when the prior quarter is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Revenue]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Revenue Standalone Quarter],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(_annual || ISBLANK(_curr) || ISBLANK(_prior), BLANK(), DIVIDE(_curr - _prior, _prior))"
  },
  {
    "name": "Net Income YoY Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Net Income minus the same NVIDIA fiscal quarter in the prior fiscal year (Date[fiscal_year]/[fiscal_quarter] quarter-end). Not SAMEPERIODLASTYEAR. BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Net Income]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Net Income],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Net Income YoY %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "YoY change in Net Income as a % of the prior fiscal-year same quarter. Uses fiscal calendar, not SAMEPERIODLASTYEAR.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Net Income]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Net Income],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Net Income QoQ Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Net Income minus the prior fiscal quarter (Q1 wraps to prior FY Q4, derived as FY − Q1 − Q2 − Q3 via [Net Income Standalone Quarter] so a quarter is never compared to a full year). BLANK when the latest period is the annual 10-K, or when the prior quarter is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Net Income]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Net Income Standalone Quarter],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(_annual || ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Net Income QoQ %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "QoQ change in Net Income as a % of the prior fiscal quarter (Q1 wraps to prior FY Q4, derived as FY − Q1 − Q2 − Q3 via [Net Income Standalone Quarter] so a quarter is never compared to a full year). BLANK when the latest period is the annual 10-K, or when the prior quarter is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Net Income]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Net Income Standalone Quarter],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(_annual || ISBLANK(_curr) || ISBLANK(_prior), BLANK(), DIVIDE(_curr - _prior, _prior))"
  },
  {
    "name": "Total Assets YoY Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Total Assets minus the same NVIDIA fiscal quarter in the prior fiscal year (Date[fiscal_year]/[fiscal_quarter] quarter-end). Not SAMEPERIODLASTYEAR. BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Assets]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Assets],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Total Assets YoY %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "YoY change in Total Assets as a % of the prior fiscal-year same quarter. Uses fiscal calendar, not SAMEPERIODLASTYEAR.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Assets]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Assets],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Total Assets QoQ Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Total Assets minus the prior fiscal quarter (Q1 wraps to prior FY Q4). Point-in-time; valid at fiscal year-end (Q4 vs Q3). BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Assets]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Assets],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Total Assets QoQ %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "QoQ change in Total Assets as a % of the prior fiscal quarter.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Assets]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Assets],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Equity YoY Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Equity minus the same NVIDIA fiscal quarter in the prior fiscal year (Date[fiscal_year]/[fiscal_quarter] quarter-end). Not SAMEPERIODLASTYEAR. BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Equity]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Equity],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Equity YoY %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "YoY change in Equity as a % of the prior fiscal-year same quarter. Uses fiscal calendar, not SAMEPERIODLASTYEAR.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Equity]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Equity],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Equity QoQ Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Equity minus the prior fiscal quarter (Q1 wraps to prior FY Q4). Point-in-time; valid at fiscal year-end (Q4 vs Q3). BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Equity]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Equity],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Equity QoQ %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "QoQ change in Equity as a % of the prior fiscal quarter.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Total Equity]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Total Equity],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Cash YoY Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Cash minus the same NVIDIA fiscal quarter in the prior fiscal year (Date[fiscal_year]/[fiscal_quarter] quarter-end). Not SAMEPERIODLASTYEAR. BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Cash]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Cash],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Cash YoY %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "YoY change in Cash as a % of the prior fiscal-year same quarter. Uses fiscal calendar, not SAMEPERIODLASTYEAR.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Cash]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 1,\n        'Date'[fiscal_quarter] = _fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Cash],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "Cash QoQ Change",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Cash minus the prior fiscal quarter (Q1 wraps to prior FY Q4). Point-in-time; valid at fiscal year-end (Q4 vs Q3). BLANK when the current or prior value is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Cash]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Cash],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nIF(ISBLANK(_curr) || ISBLANK(_prior), BLANK(), _curr - _prior)"
  },
  {
    "name": "Cash QoQ %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "QoQ change in Cash as a % of the prior fiscal quarter.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Balance Sheet'[period_end]),\n        'Balance Sheet'[is_latest_filing] = TRUE()\n    )\nVAR _curr = [Cash]\nVAR _fy =\n    CALCULATE(MAX('Date'[fiscal_year]), 'Date'[date] = _end)\nVAR _fq =\n    CALCULATE(MAX('Date'[fiscal_quarter]), 'Date'[date] = _end)\nVAR _prior_fy = IF(_fq = 1, _fy - 1, _fy)\nVAR _prior_fq = IF(_fq = 1, 4, _fq - 1)\nVAR _prior_end =\n    CALCULATE(\n        MAX('Date'[date]),\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _prior_fy,\n        'Date'[fiscal_quarter] = _prior_fq,\n        'Date'[is_fiscal_quarter_end] = TRUE()\n    )\nVAR _prior =\n    CALCULATE(\n        [Cash],\n        REMOVEFILTERS('Date'),\n        'Date'[date] = _prior_end\n    )\nRETURN\nDIVIDE(_curr - _prior, _prior)"
  },
  {
    "name": "TTM Revenue",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Trailing four standalone fiscal quarters of Revenue ending at the latest period in context ([Revenue Standalone Quarter], so Q4 = FY − Q1 − Q2 − Q3). BLANK unless all four quarters exist. If the current period is the annual 10-K (FY or duration > 300 days), this equals [Revenue].",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _q_ends =\n    TOPN(\n        4,\n        CALCULATETABLE(\n            VALUES('Date'[date]),\n            REMOVEFILTERS('Date'),\n            'Date'[is_fiscal_quarter_end] = TRUE(),\n            'Date'[date] <= _end\n        ),\n        'Date'[date], DESC\n    )\nVAR _quarters =\n    ADDCOLUMNS(\n        _q_ends,\n        \"@v\",\n            VAR _d = 'Date'[date]\n            RETURN CALCULATE([Revenue Standalone Quarter], REMOVEFILTERS('Date'), 'Date'[date] = _d)\n    )\nVAR _n = COUNTROWS(FILTER(_quarters, NOT ISBLANK([@v])))\nRETURN\nIF(\n    ISBLANK(_end),\n    BLANK(),\n    IF(_annual, [Revenue], IF(_n = 4, SUMX(_quarters, [@v])))\n)"
  },
  {
    "name": "TTM Net Income",
    "folder": "04-analysis",
    "format": "#,##0",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Trailing four standalone fiscal quarters of Net Income ending at the latest period in context ([Net Income Standalone Quarter], so Q4 = FY − Q1 − Q2 − Q3). BLANK unless all four quarters exist. If the current period is the annual 10-K (FY or duration > 300 days), this equals [Net Income].",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fp =\n    CALCULATE(\n        MAX('Income Statement'[fiscal_period]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _days =\n    CALCULATE(\n        MAX('Income Statement'[duration_days]),\n        'Income Statement'[is_latest_filing] = TRUE(),\n        'Income Statement'[period_end] = _end\n    )\nVAR _annual = _fp = \"FY\" || _days > 300\nVAR _q_ends =\n    TOPN(\n        4,\n        CALCULATETABLE(\n            VALUES('Date'[date]),\n            REMOVEFILTERS('Date'),\n            'Date'[is_fiscal_quarter_end] = TRUE(),\n            'Date'[date] <= _end\n        ),\n        'Date'[date], DESC\n    )\nVAR _quarters =\n    ADDCOLUMNS(\n        _q_ends,\n        \"@v\",\n            VAR _d = 'Date'[date]\n            RETURN CALCULATE([Net Income Standalone Quarter], REMOVEFILTERS('Date'), 'Date'[date] = _d)\n    )\nVAR _n = COUNTROWS(FILTER(_quarters, NOT ISBLANK([@v])))\nRETURN\nIF(\n    ISBLANK(_end),\n    BLANK(),\n    IF(_annual, [Net Income], IF(_n = 4, SUMX(_quarters, [@v])))\n)"
  },
  {
    "name": "Revenue CAGR 3Y",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "3-year compound annual growth of fiscal-year Revenue for the latest COMPLETED fiscal year on or before the latest period in context: (Rev_FY / Rev_FY-3)^(1/3) − 1. An in-progress year (latest period is a 10-Q) uses the prior completed FY. BLANK when either year is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(\n        MAX('Date'[fiscal_year]),\n        REMOVEFILTERS('Date'),\n        'Date'[is_fiscal_year_end] = TRUE(),\n        'Date'[date] <= _end\n    )\nVAR _curr =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nVAR _base =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 3,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nRETURN\nIF(\n    ISBLANK(_curr) || ISBLANK(_base) || _base <= 0,\n    BLANK(),\n    POWER(DIVIDE(_curr, _base), 1 / 3) - 1\n)"
  },
  {
    "name": "Revenue CAGR 5Y",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "5-year compound annual growth of fiscal-year Revenue for the latest COMPLETED fiscal year on or before the latest period in context: (Rev_FY / Rev_FY-5)^(1/5) − 1. An in-progress year (latest period is a 10-Q) uses the prior completed FY. BLANK when either year is missing.",
    "expr": "VAR _end =\n    CALCULATE(\n        MAX('Income Statement'[period_end]),\n        'Income Statement'[is_latest_filing] = TRUE()\n    )\nVAR _fy =\n    CALCULATE(\n        MAX('Date'[fiscal_year]),\n        REMOVEFILTERS('Date'),\n        'Date'[is_fiscal_year_end] = TRUE(),\n        'Date'[date] <= _end\n    )\nVAR _curr =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nVAR _base =\n    CALCULATE(\n        [Revenue],\n        REMOVEFILTERS('Date'),\n        'Date'[fiscal_year] = _fy - 5,\n        'Date'[is_fiscal_year_end] = TRUE()\n    )\nRETURN\nIF(\n    ISBLANK(_curr) || ISBLANK(_base) || _base <= 0,\n    BLANK(),\n    POWER(DIVIDE(_curr, _base), 1 / 5) - 1\n)"
  },
  {
    "name": "Common Size %",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "Selected Line Item amount at the latest period_end (is_latest_filing) as a % of Revenue (income statement and cash flow) or Total Assets (balance sheet). Requires a single selected line item.",
    "expr": "VAR _stmt = SELECTEDVALUE('Line Item'[statement])\nVAR _concept = SELECTEDVALUE('Line Item'[concept])\nVAR _end =\n    SWITCH(\n        _stmt,\n        \"balance_sheet\",\n            CALCULATE(MAX('Balance Sheet'[period_end]), 'Balance Sheet'[is_latest_filing] = TRUE()),\n        \"income_statement\",\n            CALCULATE(MAX('Income Statement'[period_end]), 'Income Statement'[is_latest_filing] = TRUE()),\n        \"cash_flow\",\n            CALCULATE(MAX('Cash Flow'[period_end]), 'Cash Flow'[is_latest_filing] = TRUE())\n    )\nVAR _amt =\n    SWITCH(\n        _stmt,\n        \"balance_sheet\",\n            CALCULATE(\n                SUM('Balance Sheet'[value_usd]),\n                'Balance Sheet'[concept] = _concept,\n                'Balance Sheet'[is_latest_filing] = TRUE(),\n                'Balance Sheet'[period_end] = _end\n            ),\n        \"income_statement\",\n            CALCULATE(\n                SUM('Income Statement'[value_usd]),\n                'Income Statement'[concept] = _concept,\n                'Income Statement'[is_latest_filing] = TRUE(),\n                'Income Statement'[period_end] = _end\n            ),\n        \"cash_flow\",\n            CALCULATE(\n                SUM('Cash Flow'[value_usd]),\n                'Cash Flow'[concept] = _concept,\n                'Cash Flow'[is_latest_filing] = TRUE(),\n                'Cash Flow'[period_end] = _end\n            )\n    )\nVAR _base =\n    SWITCH(\n        _stmt,\n        \"balance_sheet\", [Total Assets],\n        \"income_statement\", [Revenue],\n        \"cash_flow\", [Revenue]\n    )\nRETURN\nDIVIDE(_amt, _base)"
  },
  {
    "name": "DuPont ROE",
    "folder": "04-analysis",
    "format": "0.0%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "DuPont decomposition: Net Margin × Asset Turnover × Equity Multiplier. With average balances this equals [ROE].",
    "expr": "[Net Margin %] * [Asset Turnover] * [Equity Multiplier]"
  },
  {
    "name": "DuPont Check",
    "folder": "04-analysis",
    "format": "0.0000%",
    "table": "Balance Sheet",
    "hidden": false,
    "description": "DuPont ROE minus ROE. Should be ~0 when averages are used consistently. A material value signals a filter or missing-balance issue.",
    "expr": "[DuPont ROE] - [ROE]"
  }
]''')
OWNED = {m["name"] for m in MEASURES}
print("owned measures:", len(OWNED))
print("lakehouse attached:", lakehouse_attached())
if not lakehouse_attached():
    print("WARN: attach NVIDIALakehouse — the independent gold check needs /lakehouse/default/Tables/gold/…")


## 1 · Independent FY totals from gold Delta (not DAX)


In [ ]:
def read_gold(name: str) -> pd.DataFrame:
    candidates = [
        Path(f"/lakehouse/default/Tables/gold/{name}"),
        Path(f"/lakehouse/default/Tables/{name}"),
    ]
    last_err = None
    for p in candidates:
        if not p.exists():
            continue
        try:
            from deltalake import DeltaTable

            dt = DeltaTable(str(p))
            print(f"gold.{name}: delta version {dt.version()} via mount")
            return dt.to_pandas()
        except Exception as ex:
            last_err = ex
        try:
            # duckdb's delta extension reads deletion vectors (Fabric Spark enables them by default)
            import duckdb

            con = duckdb.connect()
            con.sql("INSTALL delta; LOAD delta;")
            out = con.sql(f"SELECT * FROM delta_scan('{p}')").df()
            print(f"gold.{name}: read via duckdb delta_scan")
            return out
        except Exception as ex:
            last_err = ex
        # No pd.read_parquet fallback: a Delta folder keeps files from overwritten versions,
        # so reading raw parquet returns stale rows mixed with current ones.
        print(f"deltalake could not read {p}: {last_err!r}; trying OneLake")
    # Fallback: read straight from OneLake (works even when the default-lakehouse mount is missing)
    try:
        from deltalake import DeltaTable
        import notebookutils

        uri = (
            "abfss://d1d09fde-24c7-4984-a4b0-eb55acf7c4f4@onelake.dfs.fabric.microsoft.com/"
            f"ece4bf8e-457a-4e4e-9628-4957de7a9c54/Tables/gold/{name}"
        )
        opts = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
        dt = DeltaTable(uri, storage_options=opts)
        print(f"gold.{name}: delta version {dt.version()} via OneLake")
        return dt.to_pandas()
    except Exception as ex:
        last_err = ex
    raise FileNotFoundError(f"gold.{name} not readable from mount or OneLake (last error: {last_err})")


def latest(df: pd.DataFrame, concept: str, fy: int, fp: str = "FY"):
    hit = df[
        (df["concept"] == concept)
        & (df["fiscal_year"] == fy)
        & (df["fiscal_period"] == fp)
        & (df["is_latest_filing"] == True)
    ]
    if hit.empty:
        return None
    return float(hit.iloc[0]["value_usd"])


def revenue(inc: pd.DataFrame, fy: int, fp: str = "FY"):
    return latest(inc, "RevenueFromContractWithCustomerExcludingAssessedTax", fy, fp) or latest(
        inc, "Revenues", fy, fp
    )


def cogs(inc: pd.DataFrame, fy: int, fp: str = "FY"):
    return latest(inc, "CostOfRevenue", fy, fp) or latest(inc, "CostOfGoodsAndServicesSold", fy, fp)


bs = read_gold("balance_sheet")
inc = read_gold("income_statement")
cf = read_gold("cash_flow")
print("rows BS/IS/CF:", len(bs), len(inc), len(cf))
fy = FY_CHECK
prior = fy - 1

rev = revenue(inc, fy)
rev_prior = revenue(inc, prior)
ni = latest(inc, "NetIncomeLoss", fy)
gp = latest(inc, "GrossProfit", fy)
oi = latest(inc, "OperatingIncomeLoss", fy)
rd = latest(inc, "ResearchAndDevelopmentExpense", fy)
cost = cogs(inc, fy)
eq = latest(bs, "StockholdersEquity", fy)
eq_p = latest(bs, "StockholdersEquity", prior)
assets = latest(bs, "Assets", fy)
assets_p = latest(bs, "Assets", prior)
ca = latest(bs, "AssetsCurrent", fy)
cl = latest(bs, "LiabilitiesCurrent", fy)
cash = latest(bs, "CashAndCashEquivalentsAtCarryingValue", fy)
ms = latest(bs, "MarketableSecuritiesCurrent", fy)
ar = latest(bs, "AccountsReceivableNetCurrent", fy)
ar_p = latest(bs, "AccountsReceivableNetCurrent", prior)
inv = latest(bs, "InventoryNet", fy)
inv_p = latest(bs, "InventoryNet", prior)
liab = latest(bs, "Liabilities", fy)
ltd = latest(bs, "LongTermDebtNoncurrent", fy)
cfo = latest(cf, "NetCashProvidedByUsedInOperatingActivities", fy)
days_hit = inc[
    (inc["fiscal_year"] == fy)
    & (inc["fiscal_period"] == "FY")
    & (inc["is_latest_filing"] == True)
    & (inc["concept"].isin(["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "NetIncomeLoss"]))
]
days = float(days_hit["duration_days"].dropna().iloc[0]) if not days_hit["duration_days"].dropna().empty else 365.0

def avg(curr, prior_v):
    if curr is None:
        return None
    if prior_v is None:
        return curr
    return (curr + prior_v) / 2.0

avg_eq = avg(eq, eq_p)
avg_a = avg(assets, assets_p)
avg_inv = avg(inv, inv_p)
avg_ar = avg(ar, ar_p)
ms_q = 0.0 if ms is None else ms

EXPECTED = {
    "Gross Margin %": None if not (gp and rev) else gp / rev,
    "Operating Margin %": None if not (oi and rev) else oi / rev,
    "Net Margin %": None if not (ni and rev) else ni / rev,
    "ROE": None if not (ni and avg_eq) else ni / avg_eq,
    "ROA": None if not (ni and avg_a) else ni / avg_a,
    "Current Ratio": None if not (ca and cl) else ca / cl,
    "Debt to Equity": None if not (ltd and eq) else ltd / eq,
    "Revenue YoY %": None if not (rev and rev_prior) else (rev - rev_prior) / rev_prior,
    "TTM Revenue": rev,
    "DuPont ROE": None
    if not (ni and rev and avg_a and avg_eq)
    else (ni / rev) * (rev / avg_a) * (avg_a / avg_eq),
}

print(f"Independent gold FY{fy} (latest filing, annual):")
print(f"  Revenue={rev:,.0f}  prior FY={rev_prior:,.0f}  Net Income={ni:,.0f}")
print(f"  Assets={assets:,.0f} / prior={assets_p:,.0f}  avg={avg_a:,.0f}")
print(f"  Equity={eq:,.0f} / prior={eq_p:,.0f}  avg={avg_eq:,.0f}")
print(f"  Period days={days}  Marketable securities={ms}")
for k, v in EXPECTED.items():
    print(f"  {k}: {v}")

# ----- Quarter-grain checks (independent pandas from gold vs DAX) -----
def _rev_rows(df):
    m = df["concept"].isin(["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues"]) & (df["is_latest_filing"] == True)
    out = df[m].copy()
    out["_pref"] = (out["concept"] == "RevenueFromContractWithCustomerExcludingAssessedTax").astype(int)
    out = out.sort_values("_pref").drop_duplicates(["fiscal_year", "fiscal_period"], keep="last")
    return {(int(r.fiscal_year), r.fiscal_period): float(r.value_usd) for r in out.itertuples()}


def _standalone_q(vals, fy, q):
    if q < 4:
        return vals.get((fy, f"Q{q}"))
    parts = [vals.get((fy, p)) for p in ("FY", "Q1", "Q2", "Q3")]
    return None if any(v is None for v in parts) else parts[0] - parts[1] - parts[2] - parts[3]


def _prev_q(fy, q):
    return (fy - 1, 4) if q == 1 else (fy, q - 1)


REV = _rev_rows(inc)
q_rows = inc[(inc["is_latest_filing"] == True) & inc["fiscal_period"].isin(["Q1", "Q2", "Q3"])]
LAT_FY = int(q_rows["fiscal_year"].max())
LAT_Q = int(q_rows[q_rows["fiscal_year"] == LAT_FY]["fiscal_period"].str[1].astype(int).max())
Q1_FY = int(q_rows[q_rows["fiscal_period"] == "Q1"]["fiscal_year"].max())

assert not q_rows["is_ytd"].astype(bool).any(), "gold still flags YTD income rows as latest on Q1-Q3 (rerun the gold notebook)"

ttm = [_standalone_q(REV, *qq) for qq in [(LAT_FY, LAT_Q), _prev_q(LAT_FY, LAT_Q), _prev_q(*_prev_q(LAT_FY, LAT_Q)), _prev_q(*_prev_q(*_prev_q(LAT_FY, LAT_Q)))]]
q4_prior = _standalone_q(REV, Q1_FY - 1, 4)
last_fy = max(fy_ for (fy_, fp) in REV if fp == "FY")
Q_EXPECTED = {
    f"Revenue Q{LAT_Q} FY{LAT_FY}": REV.get((LAT_FY, f"Q{LAT_Q}")),
    f"TTM Revenue Q{LAT_Q} FY{LAT_FY}": None if any(v is None for v in ttm) else sum(ttm),
    f"Revenue QoQ % Q1 FY{Q1_FY}": None if not q4_prior else (REV[(Q1_FY, "Q1")] - q4_prior) / q4_prior,
    "Revenue CAGR 3Y (no filter)": (REV[(last_fy, "FY")] / REV[(last_fy - 3, "FY")]) ** (1 / 3) - 1,
}
print("Quarter-grain expectations from gold:")
for k, v in Q_EXPECTED.items():
    print(f"  {k}: {v}")


## 2 · Upsert owned measures via TOM


In [ ]:
def upsert_measure(tom, host: str, m: dict) -> None:
    current = next((meas for meas in tom.all_measures() if meas.Name == m["name"]), None)
    if current is not None and current.Parent.Name != host:
        if current.Name not in OWNED:
            raise SystemExit(f"Refusing to move unowned measure '{current.Name}' from {current.Parent.Name}")
        tom.remove_object(current)
        current = None
    if current is not None:
        tom.update_measure(
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )
    else:
        tom.add_measure(
            table_name=host,
            measure_name=m["name"],
            expression=m["expr"],
            format_string=m["format"],
            hidden=m["hidden"],
            description=m["description"],
            display_folder=m["folder"],
        )


have_host = False
with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    names = [t.Name for t in tom.model.Tables]
    if HOST not in names:
        raise SystemExit(f"Host table '{HOST}' not in {names}. Run nb_semantic_model first.")
    have_host = True
    for m in MEASURES:
        upsert_measure(tom, HOST, m)
        print("upsert", m["folder"], m["name"])
print("TOM saved")


## 3 · Frame, FY2026 + quarter-grain DAX vs gold, error sweep


In [ ]:
refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")
print("refresh/framing complete")


def eval_row(pairs):
    args = ",\n    ".join(
        '"{label}", CALCULATE({expr}, \'Date\'[fiscal_year] = {fy})'.format(
            label=label, expr=expr, fy=FY_CHECK
        )
        for label, expr in pairs
    )
    dax = "EVALUATE\nROW(\n    " + args + "\n)"
    return fabric.evaluate_dax(dataset=DATASET, workspace=WORKSPACE, dax_string=dax)


def col_of(df, label):
    return next((c for c in df.columns if label in str(c)), None)


def close(got, exp, dollars=False):
    if got is None or (isinstance(got, float) and pd.isna(got)):
        return exp is None
    if exp is None:
        return False
    if dollars:
        return abs(float(got) - float(exp)) <= 1.0
    return abs(float(got) - float(exp)) <= max(1e-6, 1e-6 * abs(float(exp)))


failed = []

check_pairs = [
    ("Gross Margin %", "[Gross Margin %]"),
    ("Operating Margin %", "[Operating Margin %]"),
    ("Net Margin %", "[Net Margin %]"),
    ("ROE", "[ROE]"),
    ("ROA", "[ROA]"),
    ("Current Ratio", "[Current Ratio]"),
    ("Debt to Equity", "[Debt to Equity]"),
    ("Revenue YoY %", "[Revenue YoY %]"),
    ("TTM Revenue", "[TTM Revenue]"),
    ("DuPont ROE", "[DuPont ROE]"),
]
result = eval_row(check_pairs)
print(result)
row = result.iloc[0]

for label, _ in check_pairs:
    c = col_of(result, label)
    got = None if c is None else row[c]
    exp = EXPECTED[label]
    dollars = label == "TTM Revenue"
    ok = c is not None and close(got, exp, dollars=dollars)
    print(f"  {'PASS' if ok else 'FAIL'} {label}: dax={got} gold={exp}")
    if not ok:
        failed.append(label)

c_dup = col_of(result, "DuPont ROE")
c_roe = col_of(result, "ROE")
if c_dup is None or c_roe is None or not close(row[c_dup], row[c_roe]):
    print(
        f"  FAIL DuPont product == ROE: "
        f"{None if c_dup is None else row[c_dup]} vs {None if c_roe is None else row[c_roe]}"
    )
    failed.append("DuPont product == ROE")
else:
    print(f"  PASS DuPont product == ROE ({row[c_dup]})")

def eval_ctx(label, expr, ctx):
    filt = "".join(f", {c}" for c in ctx)
    dax = f'EVALUATE ROW("{label}", CALCULATE({expr}{filt}))'
    df = fabric.evaluate_dax(dataset=DATASET, workspace=WORKSPACE, dax_string=dax)
    c = next((c for c in df.columns if str(c).endswith(f"[{label}]") or str(c) == label), df.columns[0])
    return df.iloc[0][c]


q_checks = [
    (f"Revenue Q{LAT_Q} FY{LAT_FY}", "[Revenue]", [f"'Date'[fiscal_year] = {LAT_FY}", f"'Date'[fiscal_quarter] = {LAT_Q}"], True),
    (f"TTM Revenue Q{LAT_Q} FY{LAT_FY}", "[TTM Revenue]", [f"'Date'[fiscal_year] = {LAT_FY}", f"'Date'[fiscal_quarter] = {LAT_Q}"], True),
    (f"Revenue QoQ % Q1 FY{Q1_FY}", "[Revenue QoQ %]", [f"'Date'[fiscal_year] = {Q1_FY}", "'Date'[fiscal_quarter] = 1"], False),
    ("Revenue CAGR 3Y (no filter)", "[Revenue CAGR 3Y]", [], False),
]
print("\nQuarter-grain checks (DAX vs gold):")
for label, expr, ctx, dollars in q_checks:
    try:
        got = eval_ctx("v", expr, ctx)
    except Exception as ex:
        got = None
        print(f"  FAIL {label}: {type(ex).__name__}: {ex}")
    exp = Q_EXPECTED[label]
    ok = close(got, exp, dollars=dollars)
    print(f"  {'PASS' if ok else 'FAIL'} {label}: dax={got} gold={exp}")
    if not ok:
        failed.append(label)

print("\nError sweep (FY2026):")
for m in MEASURES:
    name = m["name"]
    try:
        df = eval_row([(name, f"[{name}]")])
        err_cols = [c for c in df.columns if "error" in str(c).lower()]
        if err_cols and pd.notna(df.iloc[0][err_cols[0]]) and str(df.iloc[0][err_cols[0]]).strip():
            print(f"  FAIL {name}: {df.iloc[0][err_cols[0]]}")
            failed.append(name)
        else:
            c = col_of(df, name)
            val = None if c is None else df.iloc[0][c]
            print(f"  ok {name}={val}")
    except Exception as ex:
        print(f"  FAIL {name}: {type(ex).__name__}: {ex}")
        failed.append(name)

print("OVERALL:", "FAIL" if failed else "PASS")
if failed:
    raise AssertionError("failed: " + ", ".join(failed))
